# Raw video VideoMAE RunPod 실험 - 라벨별 50개

10초 원본 영상을 clip으로 자르지 않고 그대로 사용해 VideoMAE를 학습합니다.

실험 범위:
- 상위 사고 유형별 50개
- 총 200개 영상 기준
- train/val/test = 70/20/10
- frame_count=16

실험 조합:
1. freeze_backbone=True, lr=1e-4
2. freeze_backbone=False, lr=1e-5
3. freeze_backbone=False, lr=1e-5, epoch=50

주의:
- 이 노트북은 RunPod의 `/workspace/SKN27-FINAL-3Team` 경로를 기준으로 실행합니다.
- raw 영상이 이미 있으면 재사용하고, 없으면 manifest 기준으로 다시 다운로드합니다.


In [8]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
from collections import Counter, defaultdict

RUN_INSTALL_REQUIREMENTS = True
RUN_DOWNLOAD = True
RUN_TRAIN_EXP1 = True
RUN_TRAIN_EXP2 = True
RUN_TRAIN_EXP3 = True

PROJECT_ROOT = Path('/workspace/SKN27-FINAL-3Team')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'RunPod project root not found: {PROJECT_ROOT}')

MANIFEST_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests'
RAW_VIDEO_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/raw_videos'
MODEL_DIR = PROJECT_ROOT / 'storage/vision/models/videomae_raw_video'
SAMPLE_MANIFEST = MANIFEST_DIR / 'sample_700_coarse_manifest.csv'
FULL_DOWNLOAD_MANIFEST = MANIFEST_DIR / 'train_700_download_manifest.csv'
MANIFEST_50 = MANIFEST_DIR / 'train_50_raw_video_manifest.csv'
SPLIT_MANIFEST_50 = MANIFEST_DIR / 'train_50_raw_video_manifest_split.csv'

FRAME_COUNT = 16
BATCH_SIZE = 1
EPOCHS = 5
SEED = 42
DEVICE = 'auto'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FULL_DOWNLOAD_MANIFEST:', FULL_DOWNLOAD_MANIFEST)
print('SPLIT_MANIFEST_50:', SPLIT_MANIFEST_50)


PROJECT_ROOT: D:\dev\SKN27-FINAL-3Team
FULL_DOWNLOAD_MANIFEST: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_700_download_manifest.csv
SPLIT_MANIFEST_50: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv


In [9]:
def run_command(command, *, enabled=True, timeout=None):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    if not enabled:
        print('SKIPPED')
        return None
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding='utf-8',
        errors='replace',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    try:
        for line in process.stdout:
            print(line, end='')
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill()
        raise
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def resolve_local_video_path(path_value):
    if not path_value:
        return None
    path_text = str(path_value).replace('\\', '/')
    runpod_prefix = '/workspace/SKN27-FINAL-3Team/'
    if path_text.startswith(runpod_prefix):
        return PROJECT_ROOT / path_text[len(runpod_prefix):]
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def manifest_local_path(row):
    return resolve_local_video_path(row.get('local_path') or row.get('file_path'))


def manifest_has_missing_videos(path):
    if not Path(path).exists():
        return True
    rows = read_csv(path)
    if not rows:
        return True
    return any((manifest_local_path(row) is None or not manifest_local_path(row).exists()) for row in rows)


def ensure_source_manifests():
    DRIVE_LISTING = PROJECT_ROOT / 'storage/vision/manifests/drive_listing_aihub.json'
    CLASSIFICATION_MANIFEST = MANIFEST_DIR / 'classification_manifest.csv'

    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

    if not CLASSIFICATION_MANIFEST.exists():
        if not DRIVE_LISTING.exists():
            raise FileNotFoundError(
                'Missing both classification manifest and Drive listing. Upload one of these files first: '
                f'{CLASSIFICATION_MANIFEST} or {DRIVE_LISTING}'
            )
        run_command([
            sys.executable,
            'etl/vision/build_classification_manifest.py',
            '--listing', DRIVE_LISTING,
            '--output', CLASSIFICATION_MANIFEST,
        ], enabled=True, timeout=1800)

    if not SAMPLE_MANIFEST.exists():
        run_command([
            sys.executable,
            'etl/vision/sample_classification_dataset.py',
            '--input', CLASSIFICATION_MANIFEST,
            '--output', SAMPLE_MANIFEST,
            '--label-column', 'coarse_label',
            '--per-label', '700',
            '--seed', str(SEED),
            '--train-ratio', '0.7',
            '--val-ratio', '0.2',
        ], enabled=True, timeout=1800)

    return SAMPLE_MANIFEST

def ensure_download_manifest():
    ensure_source_manifests()

    needs_download = manifest_has_missing_videos(FULL_DOWNLOAD_MANIFEST)
    if needs_download and not SAMPLE_MANIFEST.exists():
        raise FileNotFoundError(f'Raw videos are missing and source sample manifest is missing: {SAMPLE_MANIFEST}')

    if needs_download:
        run_command([
            sys.executable,
            'etl/vision/download_sampled_media.py',
            '--input', SAMPLE_MANIFEST,
            '--output', FULL_DOWNLOAD_MANIFEST,
            '--download-dir', RAW_VIDEO_DIR,
            '--label-column', 'coarse_label',
            '--per-label', '700',
            '--split', '',
        ], enabled=RUN_DOWNLOAD, timeout=None)

    rows = read_csv(FULL_DOWNLOAD_MANIFEST)
    print('download_rows:', len(rows))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in rows)))
    print('download_status:', dict(Counter(row.get('download_status') for row in rows)))
    print('file_exists:', dict(Counter(str((manifest_local_path(row) or Path()).exists()) for row in rows)))
    return FULL_DOWNLOAD_MANIFEST


def make_subset_manifest(source_manifest, output_manifest, per_label):
    rows = read_csv(source_manifest)
    selected = []
    counts = defaultdict(int)
    missing = 0
    for row in rows:
        label = row.get('coarse_label') or row.get('label')
        if not label or counts[label] >= per_label:
            continue
        path = manifest_local_path(row)
        if path is None or not path.exists():
            missing += 1
            continue
        copied = dict(row)
        copied['local_path'] = str(path)
        copied['file_exists'] = 'True'
        selected.append(copied)
        counts[label] += 1
    write_csv(selected, output_manifest)
    print('subset_manifest:', output_manifest)
    print('rows:', len(selected))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in selected)))
    print('skipped_missing_videos:', missing)
    if not selected:
        raise FileNotFoundError('No local videos found. Check download manifest and raw_videos directory.')
    return output_manifest


def make_split_manifest(source_manifest, output_manifest, train_ratio=0.7, val_ratio=0.2):
    rows = read_csv(source_manifest)
    grouped = defaultdict(list)
    for row in rows:
        grouped[row.get('coarse_label')].append(row)
    split_rows = []
    for label, label_rows in grouped.items():
        total = len(label_rows)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)
        for index, row in enumerate(label_rows):
            copied = dict(row)
            if index < train_end:
                copied['split'] = 'train'
            elif index < val_end:
                copied['split'] = 'val'
            else:
                copied['split'] = 'test'
            split_rows.append(copied)
    write_csv(split_rows, output_manifest)
    print('split_manifest:', output_manifest)
    print('rows:', len(split_rows))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in split_rows)))
    print('split_counts:', dict(Counter(row.get('split') for row in split_rows)))
    print('label_split_counts:')
    for key, value in sorted(Counter((row.get('coarse_label'), row.get('split')) for row in split_rows).items()):
        print(key, value)
    return output_manifest


def latest_run_dir(path):
    runs = [p for p in Path(path).glob('videomae_cls_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No runs found under {path}')
    return sorted(runs)[-1]


def build_train_command(experiment):
    command = [
        sys.executable,
        'ai/vision/train_videomae_classifier.py',
        '--manifest', experiment['manifest'],
        '--root-dir', PROJECT_ROOT,
        '--output-dir', experiment['output_dir'],
        '--label-column', 'coarse_label',
        '--frame-count', experiment['frame_count'],
        '--epochs', experiment['epochs'],
        '--batch-size', experiment['batch_size'],
        '--learning-rate', experiment['learning_rate'],
        '--weight-decay', experiment['weight_decay'],
        '--early-stopping-patience', experiment['early_stopping_patience'],
        '--seed', SEED,
        '--device', DEVICE,
        '--num-workers', '0',
        '--no-show-progress',
    ]
    if experiment['freeze_backbone']:
        command.append('--freeze-backbone')
    return command


def run_experiment(experiment, *, enabled=True):
    print('\n##', experiment['name'])
    print(json.dumps({k: str(v) for k, v in experiment.items()}, ensure_ascii=False, indent=2))
    run_command(build_train_command(experiment), enabled=enabled, timeout=None)
    if enabled:
        print('LAST_RUN_DIR:', latest_run_dir(experiment['output_dir']))


In [10]:
ensure_download_manifest()


download_rows: 2800
label_counts: {'차대보행자': 700, '차대이륜차': 700, '차대자전거': 700, '차대차': 700}
download_status: {'exists': 921, 'downloaded': 1879}
file_exists: {'True': 2800}


In [11]:
make_subset_manifest(FULL_DOWNLOAD_MANIFEST, MANIFEST_50, per_label=50)
make_split_manifest(MANIFEST_50, SPLIT_MANIFEST_50)


subset_manifest: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest.csv
rows: 200
label_counts: {'차대보행자': 50, '차대이륜차': 50, '차대자전거': 50, '차대차': 50}
split_manifest: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv
rows: 200
label_counts: {'차대보행자': 50, '차대이륜차': 50, '차대자전거': 50, '차대차': 50}
split_counts: {'train': 140, 'val': 40, 'test': 20}
label_split_counts:
('차대보행자', 'test') 5
('차대보행자', 'train') 35
('차대보행자', 'val') 10
('차대이륜차', 'test') 5
('차대이륜차', 'train') 35
('차대이륜차', 'val') 10
('차대자전거', 'test') 5
('차대자전거', 'train') 35
('차대자전거', 'val') 10
('차대차', 'test') 5
('차대차', 'train') 35
('차대차', 'val') 10


WindowsPath('D:/dev/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_50_raw_video_manifest_split.csv')

In [12]:
EXPERIMENT_1 = {
    'name': 'exp1_raw50_freeze_lr1e-4_fc16',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp1_freeze_lr1e-4',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.0001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': True,
}
run_experiment(EXPERIMENT_1, enabled=RUN_TRAIN_EXP1)



## exp1_raw50_freeze_lr1e-4_fc16
{
  "name": "exp1_raw50_freeze_lr1e-4_fc16",
  "manifest": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\datasets\\classification\\manifests\\train_50_raw_video_manifest_split.csv",
  "output_dir": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\models\\videomae_raw_video\\per_label_50_exp1_freeze_lr1e-4",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "0.0001",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "True"
}
$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/train_videomae_classifier.py --manifest D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp1_freeze_lr1e-4 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 0.0001 --weight-decay 0.05 --ear

In [13]:
EXPERIMENT_2 = {
    'name': 'exp2_raw50_unfreeze_lr1e-5_fc16',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp2_unfreeze_lr1e-5',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}
run_experiment(EXPERIMENT_2, enabled=RUN_TRAIN_EXP2)



## exp2_raw50_unfreeze_lr1e-5_fc16
{
  "name": "exp2_raw50_unfreeze_lr1e-5_fc16",
  "manifest": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\datasets\\classification\\manifests\\train_50_raw_video_manifest_split.csv",
  "output_dir": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\models\\videomae_raw_video\\per_label_50_exp2_unfreeze_lr1e-5",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "1e-05",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "False"
}
$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/train_videomae_classifier.py --manifest D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp2_unfreeze_lr1e-5 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 1e-05 --weight-decay 0.0

In [14]:
EXPERIMENT_3 = {
    'name': 'exp3_raw50_unfreeze_lr1e-5_fc16_e50',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp3_unfreeze_lr1e-5_e50',
    'epochs': 50,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}
run_experiment(EXPERIMENT_3, enabled=RUN_TRAIN_EXP3)


## exp3_raw50_unfreeze_lr1e-5_fc16_e50
{
  "name": "exp3_raw50_unfreeze_lr1e-5_fc16_e50",
  "manifest": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\datasets\\classification\\manifests\\train_50_raw_video_manifest_split.csv",
  "output_dir": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\models\\videomae_raw_video\\per_label_50_exp3_unfreeze_lr1e-5_e50",
  "epochs": "50",
  "batch_size": "1",
  "learning_rate": "1e-05",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "False"
}
$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/train_videomae_classifier.py --manifest D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp3_unfreeze_lr1e-5_e50 --label-column coarse_label --frame-count 16 --epochs 50 --batch-size 1 --learning-rate 1e-05 

In [15]:
def print_latest_history(output_dir):
    run_dir = latest_run_dir(output_dir)
    print('run_dir:', run_dir)
    for file_name in ['run_config.json', 'training_history.csv']:
        path = run_dir / file_name
        print('##', file_name, path.exists())
        if path.suffix == '.json' and path.exists():
            print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:3000])
        elif path.exists():
            for row in read_csv(path):
                print(row)

if RUN_TRAIN_EXP1:
    print('## EXPERIMENT_1 result')
    print_latest_history(EXPERIMENT_1['output_dir'])
if RUN_TRAIN_EXP2:
    print('## EXPERIMENT_2 result')
    print_latest_history(EXPERIMENT_2['output_dir'])
if RUN_TRAIN_EXP3:
    print('## EXPERIMENT_3 result')
    print_latest_history(EXPERIMENT_3['output_dir'])


## EXPERIMENT_1 result
run_dir: D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp1_freeze_lr1e-4\videomae_cls_20260713_182119
## run_config.json True
{
  "run_id": "videomae_cls_20260713_182119",
  "manifest": "D:/dev/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_50_raw_video_manifest_split.csv",
  "label_column": "coarse_label",
  "model_name": "MCG-NJU/videomae-base-finetuned-kinetics",
  "freeze_backbone": true,
  "frame_count": 16,
  "epochs": 5,
  "batch_size": 1,
  "learning_rate": 0.0001,
  "weight_decay": 0.05,
  "early_stopping_patience": 2,
  "best_epoch": 4,
  "best_val_accuracy": 0.4,
  "seed": 42,
  "device": "cpu",
  "train_rows": 140,
  "val_rows": 40,
  "test_rows": 20,
  "model_path": "D:/dev/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_50_exp1_freeze_lr1e-4/videomae_cls_20260713_182119"
}
## training_history.csv True
{'epoch': '1', 'train_loss': '1.384887', 'train_accuracy': '0.292857'

In [16]:
# EXPORT_ANALYSIS_ARTIFACTS
import subprocess
import sys

reports_dir = PROJECT_ROOT / 'storage/vision/reports'
command = [
    sys.executable,
    'ai/vision/export_analysis_artifacts.py',
    '--root-dir', str(PROJECT_ROOT),
    '--output-dir', str(reports_dir),
]
print('$', ' '.join(command))
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=600)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()
print('tables:', reports_dir / 'tables')
print('figures:', reports_dir / 'figures')
print('appendix:', reports_dir / 'appendix')


$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/export_analysis_artifacts.py --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\reports
tables: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\tables
figures: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\figures
appendix: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\appendix

tables: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\tables
figures: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\figures
appendix: D:\dev\SKN27-FINAL-3Team\storage\vision\reports\appendix
